# Wide table de features para o modelo — Admissões mensais por UF × grupo ocupacional

Constrói `workspace.gold.ml_features_mensal_uf_cbo`, a tabela de features do modelo DemandA no grão do escopo: **UF (27) × subgrupo ocupacional CBO (cobertura da bridge) × mês (68 meses, 2021-01 a 2026-08)**, com alvo = admissões. Consome a bridge provisória CINE×CBO (notebook 034) para repartir a formação INEP no grão ocupacional.

## Decisões de método

1. **Sem vazamento (regra de ouro)** — toda feature CAGED entra com lag ≥ 1 mês; toda feature INEP entra com lag ≥ 1 ano. O modelo só vê informação que já existia no momento da previsão.
2. **Features CAGED (giro, mensal)** — lags 1/3/12 de admissões, médias móveis 3/12, desligamento e saldo defasados (r 0,69 com t+1 na EDA), salário mediano defasado (r 0,68), harmônicos de mês (sin/cos), transformação log1p nas principais (regra de ouro do projeto).
3. **Features INEP (formação, anual, defasadas)** — concluintes defasados 1–3 anos, matrículas defasadas 1 ano, ingressantes defasados 3 e 5 anos (o pipeline de formação), share de EaD e razão de contratações por matrícula. Repartição **uniforme** da área CINE entre os subgrupos CBO do par (decisão documentada; alternativa ponderada por `confianca` fica registrada como evolução).
4. **Modalidade EaD = `tipo_modalidade_ensino = 2`** — validado contra o painel: modalidade 2 tem 50,7% das matrículas de 2024, exatamente o "estoque EaD" do dashboard.
5. **OLS como screening de features** — regressão OLS pooled (log1p do alvo, com efeitos fixos de UF e de mês) para avaliar cada feature candidata: coeficiente, p-valor e **VIF** (multicolinearidade — regra "uma medida por conceito"). O screening orienta a escolha, mas **não substitui** a validação temporal em janela expansiva nem implica causalidade. Resultado gravado em `workspace.gold.ml_features_ols_screening`.
6. **Gravação idempotente e não destrutiva** — `CREATE TABLE IF NOT EXISTS` + append por `versao_transformacao` (skip se a versão já existe). Sem `DROP`/`overwrite` — a tabela é um log versionado de snapshots; o consumo filtra pela versão vigente.

In [0]:
%pip install statsmodels
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime

# Parâmetros (widgets) — sobrescritos por jobs/agendamentos sem alterar o código
dbutils.widgets.text("versao_transformacao", "0.1.0", "Versão da transformação")
dbutils.widgets.text("tabela_features", "workspace.gold.ml_features_mensal_uf_cbo", "Tabela-alvo das features")
dbutils.widgets.text("tabela_screening", "workspace.gold.ml_features_ols_screening", "Tabela-alvo do screening OLS")
dbutils.widgets.text("tabela_bridge", "workspace.gold.bridge_cine_cbo_provisoria", "Bridge CINE×CBO (fonte do mapeamento)")

VERSAO = dbutils.widgets.get("versao_transformacao")
TABELA_FEATURES = dbutils.widgets.get("tabela_features")
TABELA_SCREENING = dbutils.widgets.get("tabela_screening")
BRIDGE = dbutils.widgets.get("tabela_bridge")

SILVER_CAGED = "workspace.silver.novo_caged_movimentacoes"
FATO_CAGED_SEM = "workspace.gold.fato_caged_semestral"
FATO_INEP = "workspace.gold.fato_inep_educacao_uf_anual"
DT_PROCESSAMENTO = datetime.now()

# Janela de dados (68 meses completos — 2026S2 é parcial)
DATA_INICIO = "2021-01-01"
DATA_FIM = "2026-09-01"  # exclusivo
ANOS_INEP = [2021, 2022, 2023, 2024]

# De-para fixo e documentado: código IBGE (CAGED) → sigla (INEP)
UF_SIGLA = {
    11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
    21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL",
    28: "SE", 29: "BA", 31: "MG", 32: "ES", 33: "RJ", 35: "SP", 41: "PR",
    42: "SC", 43: "RS", 50: "MS", 51: "MT", 52: "GO", 53: "DF",
}

print(f"Versão: {VERSAO} | Features: {TABELA_FEATURES}")
print(f"Screening OLS: {TABELA_SCREENING} | Bridge: {BRIDGE}")

In [0]:
# Bridge: pares área × subgrupo com confiança (fonte da repartição)
df_bridge = spark.table(BRIDGE)
df_pares = df_bridge.select(
    "codigo_cine_area_especifica", "codigo_cbo_subgrupo", "confianca"
).distinct()

# Nº de subgrupos por área CINE — divisor da repartição uniforme
w_area = Window.partitionBy("codigo_cine_area_especifica")
df_pares_reparticao = (
    df_pares
    .withColumn("n_subgrupos_area", F.count("*").over(w_area))
)

subgrupos_da_bridge = [r["codigo_cbo_subgrupo"] for r in df_pares.select("codigo_cbo_subgrupo").distinct().collect()]

# Preflight — coberturas verificadas em fontes agregadas (baratas), não na silver linha a linha
df_caged_preflight = spark.sql(f"""
    SELECT MIN(ano) AS ano_min, MAX(ano) AS ano_max,
           ARRAY_SORT(COLLECT_SET(DISTINCT uf)) AS ufs
    FROM {FATO_CAGED_SEM}
""").collect()[0]

df_inep_preflight = spark.sql(f"""
    SELECT MIN(ano_censo) AS ano_min, MAX(ano_censo) AS ano_max,
           COUNT(DISTINCT sigla_uf) AS qtd_ufs,
           ARRAY_SORT(COLLECT_SET(tipo_modalidade_ensino)) AS modalidades
    FROM {FATO_INEP}
""").collect()[0]

print(f"CAGED (fato semestral): anos {df_caged_preflight['ano_min']}–{df_caged_preflight['ano_max']}, {len(df_caged_preflight['ufs'])} códigos de UF")
print(f"INEP (fato UF anual): anos {df_inep_preflight['ano_min']}–{df_inep_preflight['ano_max']}, {df_inep_preflight['qtd_ufs']} UFs, modalidades {df_inep_preflight['modalidades']}")
print(f"Bridge: {len(subgrupos_da_bridge)} subgrupos CBO, {df_pares.count()} pares")

# Validação abortável: premissas de fonte
# O CAGED pode trazer o código 99 (Ignorado/Exterior) além das 27 UFs do IBGE —
# fica FORA do grão do modelo (sem correspondência territorial), exclusão documentada.
ufs_validas = set(UF_SIGLA.keys())
ufs_caged = set(df_caged_preflight["ufs"])
assert ufs_caged.issubset(ufs_validas | {99}), f"CAGED: códigos de UF inesperados: {sorted(ufs_caged - ufs_validas - {99})}"
assert df_inep_preflight["qtd_ufs"] == 27, "INEP: nº de UFs inesperado"
assert set(df_inep_preflight["modalidades"]) == {1, 2}, "INEP: modalidades inesperadas (esperado 1=presencial, 2=EaD)"
assert set(ANOS_INEP).issubset(set(range(df_inep_preflight["ano_min"], df_inep_preflight["ano_max"] + 1))), "INEP: anos do Censo ausentes"
print(f"Preflight das fontes: OK (UF 99/Ignorado será excluída do grão — exclusão documentada).")

In [0]:
# Alvo: admissões mensais por UF × subgrupo CBO (da silver, agregado em Spark)
# A filtragem por subgrupo (broadcast da bridge) acontece ANTES da agregação para reduzir o volume.
df_subgrupos_bc = F.broadcast(
    spark.createDataFrame([(s,) for s in subgrupos_da_bridge], "codigo_cbo_subgrupo STRING")
)

df_alvo = (
    spark.table(SILVER_CAGED)
    .where((F.col("competencia_mov") >= DATA_INICIO) & (F.col("competencia_mov") < DATA_FIM))
    .where(F.col("uf") != 99)  # 99 = Ignorado/Exterior — fora do grão territorial (exclusão documentada)
    .withColumn("codigo_cbo_subgrupo", F.substring(F.lpad(F.col("cbo2002_ocupacao").cast("string"), 6, "0"), 1, 3))
    .join(df_subgrupos_bc, "codigo_cbo_subgrupo", "inner")
    .groupBy("uf", "codigo_cbo_subgrupo", "competencia_mov")
    .agg(
        F.sum(F.when(F.col("saldo_movimentacao") == 1, 1).otherwise(0)).alias("admissoes"),
        F.sum(F.when(F.col("saldo_movimentacao") == -1, 1).otherwise(0)).alias("desligamentos"),
        F.percentile_approx(F.when(F.col("saldo_movimentacao") == 1, F.col("salario")), 0.5).alias("salario_mediano_admissao"),
    )
    .withColumn("sigla_uf", F.udf(lambda c: UF_SIGLA.get(c))("uf"))
    .drop("uf")
)

# Painel DENSO: mês sem movimentação = 0 admissões/desligamentos (valor legítimo no grão mensal).
# Sem isso, F.lag pegaria o mês ANTERIOR OBSERVADO (ex.: t−2) em vez do mês t−1 do calendário,
# quebrando a semântica dos lags. O salário mediano permanece nulo nesses meses (sem admissões,
# não existe mediana) — correto e tratado a jusante.
df_series = df_alvo.select("sigla_uf", "codigo_cbo_subgrupo").distinct()
df_meses = df_alvo.select("competencia_mov").distinct()
df_alvo = (
    df_series.crossJoin(df_meses)
    .join(df_alvo, ["sigla_uf", "codigo_cbo_subgrupo", "competencia_mov"], "left")
    .fillna(0, subset=["admissoes", "desligamentos"])
)

qtd_series = df_alvo.select("sigla_uf", "codigo_cbo_subgrupo").distinct().count()
qtd_meses = df_alvo.select("competencia_mov").distinct().count()
print(f"Alvo materializado (painel denso): {df_alvo.count()} linhas, {qtd_series} séries × {qtd_meses} meses")
print("Amostra do alvo:")
display(df_alvo.orderBy("competencia_mov", "sigla_uf").limit(10))

In [0]:
# Reconciliação: soma do alvo (2024) deve fechar com o fato semestral no MESMO conjunto de subgrupos
# (o total nacional difere do painel porque só entram os 26 subgrupos da bridge — esperado e documentado)

df_reconciliacao_wide = df_alvo \
    .where(F.year("competencia_mov") == 2024) \
    .groupBy("sigla_uf", "codigo_cbo_subgrupo") \
    .agg(F.sum("admissoes").alias("admissoes_wide"), F.sum("desligamentos").alias("desligamentos_wide"))

df_reconciliacao_fato = (
    spark.table(FATO_CAGED_SEM)
    .where(F.col("ano") == 2024)
    .where(F.col("uf") != 99)  # mesma exclusão aplicada ao alvo — comparação consistente
    .withColumn("codigo_cbo_subgrupo", F.substring(F.lpad(F.col("cbo2002_ocupacao").cast("string"), 6, "0"), 1, 3))
    .join(df_subgrupos_bc, "codigo_cbo_subgrupo", "inner")  # restringe aos subgrupos da bridge (escopo)
    .groupBy("uf", "codigo_cbo_subgrupo")
    .agg(F.sum("admissoes").alias("admissoes_fato"), F.sum("desligamentos").alias("desligamentos_fato"))
    .withColumn("sigla_uf", F.udf(lambda c: UF_SIGLA.get(c))("uf"))
    .drop("uf")
)

df_dif = (
    df_reconciliacao_wide
    .join(df_reconciliacao_fato, ["sigla_uf", "codigo_cbo_subgrupo"], "full_outer")
    .withColumn("dif_admissoes", F.col("admissoes_wide") - F.coalesce(F.col("admissoes_fato"), F.lit(0)))
    .withColumn("dif_desligamentos", F.col("desligamentos_wide") - F.coalesce(F.col("desligamentos_fato"), F.lit(0)))
)

divergencias = df_dif.where((F.col("dif_admissoes") != 0) | (F.col("dif_desligamentos") != 0)).count()
faltantes = df_dif.where(F.col("admissoes_wide").isNull()).count()

print(f"Pares UF × subgrupo comparados (escopo da bridge): {df_dif.count()}")
print(f"Divergências de soma: {divergencias}")
print(f"Pares presentes no fato mas ausentes no alvo (sem movimentação em nenhum mês de 2024): {faltantes}")

if divergencias > 0:
    raise ValueError("RECONCILIAÇÃO FALHOU — o alvo não fecha com o fato semestral. Abortando antes de gravar qualquer coisa.")
print("Reconciliação com o fato semestral (2024): OK — diferença zero.")

In [0]:
# Features CAGED (giro, mensal) — todas defasadas; médias móveis olham SOMENTE o passado
df_base = df_alvo \
    .withColumn("admissoes_log", F.log1p("admissoes")) \
    .withColumn("desligamentos_log", F.log1p("desligamentos")) \
    .withColumn("saldo", F.col("admissoes") - F.col("desligamentos"))

w_serie = Window.partitionBy("sigla_uf", "codigo_cbo_subgrupo").orderBy("competencia_mov")
w_serie_passado_3 = w_serie.rowsBetween(-3, -1)  # janela SEM o mês corrente (sem vazamento)
w_serie_passado_12 = w_serie.rowsBetween(-12, -1)

df_giro = (
    df_base
    .withColumn("admissoes_lag_1", F.lag("admissoes", 1).over(w_serie))
    .withColumn("admissoes_lag_3", F.lag("admissoes", 3).over(w_serie))
    .withColumn("admissoes_lag_12", F.lag("admissoes", 12).over(w_serie))
    .withColumn("admissoes_ma_3", F.avg("admissoes").over(w_serie_passado_3))
    .withColumn("admissoes_ma_12", F.avg("admissoes").over(w_serie_passado_12))
    .withColumn("desligamentos_lag_1", F.lag("desligamentos", 1).over(w_serie))
    .withColumn("desligamentos_ma_3", F.avg("desligamentos").over(w_serie_passado_3))
    .withColumn("saldo_lag_1", F.lag("saldo", 1).over(w_serie))
    .withColumn("salario_mediano_lag_1", F.lag("salario_mediano_admissao", 1).over(w_serie))
    .withColumn("admissoes_log_lag_1", F.lag("admissoes_log", 1).over(w_serie))
    .withColumn("admissoes_log_ma_3", F.avg("admissoes_log").over(w_serie_passado_3))
    .withColumn("admissoes_log_ma_12", F.avg("admissoes_log").over(w_serie_passado_12))
    # Harmônicos de mês — sazonalidade documentada na EDA (fev/ago 8,98%, dez 6,37%)
    .withColumn("mes", F.month("competencia_mov"))
    .withColumn("mes_sin", F.round(F.sin(2 * F.lit(3.141592653589793) * F.col("mes") / 12), 6))
    .withColumn("mes_cos", F.round(F.cos(2 * F.lit(3.141592653589793) * F.col("mes") / 12), 6))
)

print(f"Features de giro adicionadas: {df_giro.count()} linhas")
print("Nulls esperados só no warm-up (primeiros 12 meses de cada série):")
display(
    df_giro.select(
        [F.round(F.avg(F.when(F.col(c).isNull(), 1).otherwise(0)), 4).alias(c)
         for c in ["admissoes_lag_1", "admissoes_lag_12", "admissoes_ma_12", "salario_mediano_lag_1"]]
    )
)

In [0]:
# Features INEP (formação, anual, defasadas ≥ 1 ano) repartidas pela bridge na grão UF × subgrupo

df_inep = (
    spark.table(FATO_INEP)
    .where(F.col("ano_censo").isin(ANOS_INEP))
    .groupBy("sigla_uf", "ano_censo", "codigo_cine_area_especifica")
    .agg(
        F.sum("concluintes").alias("concluintes"),
        F.sum("matriculas").alias("matriculas"),
        F.sum("ingressantes").alias("ingressantes"),
        F.sum(F.when(F.col("tipo_modalidade_ensino") == 2, F.col("ingressantes"))).alias("ingressantes_ead"),
    )
    .withColumn("share_ead_ingresso", F.when(F.col("ingressantes") > 0, F.col("ingressantes_ead") / F.col("ingressantes")))
)

# Repartição UNIFORME: formação da área ÷ nº de subgrupos do par (decisão documentada no cabeçalho).
# ATENÇÃO: um subgrupo pode receber contribuições de VÁRIAS áreas CINE — por isso o resultado é
# AGREGADO POR SUBGRUPO (soma das contribuições repartidas), garantindo 1 linha por
# UF × subgrupo × ano. Sem isso, o join a jusante explode o painel (43,5k → 14,2M linhas).
df_inep_sub = (
    df_inep
    .join(df_pares_reparticao, "codigo_cine_area_especifica", "inner")
    .withColumn("concluintes_rep", F.round(F.col("concluintes") / F.col("n_subgrupos_area"), 2))
    .withColumn("matriculas_rep", F.round(F.col("matriculas") / F.col("n_subgrupos_area"), 2))
    .withColumn("ingressantes_rep", F.round(F.col("ingressantes") / F.col("n_subgrupos_area"), 2))
    .withColumn("ingressantes_ead_rep", F.round(F.col("ingressantes_ead") / F.col("n_subgrupos_area"), 2))
    .groupBy("sigla_uf", "ano_censo", "codigo_cbo_subgrupo")
    .agg(
        F.sum("concluintes_rep").alias("concluintes_rep"),
        F.sum("matriculas_rep").alias("matriculas_rep"),
        F.sum("ingressantes_rep").alias("ingressantes_rep"),
        F.sum("ingressantes_ead_rep").alias("ingressantes_ead_rep"),
    )
    # Share de EaD do SUBGRUPO: razão ponderada pelas contribuições dos pares (não média simples)
    .withColumn("share_ead_ingresso", F.when(F.col("ingressantes_rep") > 0, F.col("ingressantes_ead_rep") / F.col("ingressantes_rep")))
    .drop("ingressantes_ead_rep")
)

# Renomeia por defasagem e junta ao painel mensal: ano_censo = ano da competência − lag
KEYS = ["sigla_uf", "codigo_cbo_subgrupo"]

def juntar_defasado(df, df_inep_sub, coluna, defasagem, sufixo):
    df_ano = df_inep_sub.withColumnRenamed(coluna, sufixo) \
        .withColumn("ano_join", F.col("ano_censo") + F.lit(defasagem)) \
        .select(*KEYS, "ano_join", sufixo)
    return df.join(df_ano, (df.sigla_uf == df_ano.sigla_uf) &
                        (df.codigo_cbo_subgrupo == df_ano.codigo_cbo_subgrupo) &
                        (F.year(df.competencia_mov) == df_ano.ano_join), "left") \
        .drop(df_ano.sigla_uf).drop(df_ano.codigo_cbo_subgrupo).drop("ano_join")

df_formacao = df_giro
for coluna, lag_n, sufixo in [
    ("concluintes_rep", 1, "concluintes_def1"),
    ("concluintes_rep", 2, "concluintes_def2"),
    ("concluintes_rep", 3, "concluintes_def3"),
    ("matriculas_rep", 1, "matriculas_def1"),
    ("ingressantes_rep", 3, "ingressantes_def3"),
    ("ingressantes_rep", 5, "ingressantes_def5"),
    ("share_ead_ingresso", 1, "share_ead_def1"),
]:
    df_formacao = juntar_defasado(df_formacao, df_inep_sub, coluna, lag_n, sufixo)

# Feature composta: razão de contratações por matrícula (remove efeito tamanho; âncora SC 3,64 × RJ 1,76)
# Contratações do ano anterior (UF × subgrupo) ÷ matrículas defasadas 1 ano (UF × subgrupo, repartidas)
df_adm_ano = (
    df_giro.groupBy("sigla_uf", "codigo_cbo_subgrupo", F.year("competencia_mov").alias("ano"))
    .agg(F.sum("admissoes").alias("admissoes_ano"))
)
df_mat_rep = df_inep_sub.select("sigla_uf", "codigo_cbo_subgrupo", "ano_censo", "matriculas_rep")
df_razao = (
    df_adm_ano
    .join(df_mat_rep, (df_adm_ano.sigla_uf == df_mat_rep.sigla_uf) &
                     (df_adm_ano.codigo_cbo_subgrupo == df_mat_rep.codigo_cbo_subgrupo) &
                     (df_adm_ano.ano == df_mat_rep.ano_censo), "left")
    .drop(df_mat_rep.sigla_uf).drop(df_mat_rep.codigo_cbo_subgrupo)
    .withColumn("ano_join", F.col("ano") + 1)  # razão do ano t usa dados de t−1
    .withColumn("razao_contratacao_matricula", F.when(F.col("matriculas_rep") > 0, F.col("admissoes_ano") / F.col("matriculas_rep")))
    .select("sigla_uf", "codigo_cbo_subgrupo", "ano_join", "razao_contratacao_matricula")
)

df_formacao = (
    df_formacao
    .join(df_razao, (df_formacao.sigla_uf == df_razao.sigla_uf) &
                  (df_formacao.codigo_cbo_subgrupo == df_razao.codigo_cbo_subgrupo) &
                  (F.year(df_formacao.competencia_mov) == df_razao.ano_join), "left")
    .drop(df_razao.sigla_uf).drop(df_razao.codigo_cbo_subgrupo).drop("ano_join")
)

print(f"Features de formação adicionadas: {df_formacao.count()} linhas")
print("Cobertura por feature (linhas não-nulas / total) — warm-up anual esperado até 2024:")
feats_inep = ["concluintes_def1", "concluintes_def3", "matriculas_def1", "ingressantes_def5", "share_ead_def1", "razao_contratacao_matricula"]
display(df_formacao.select([F.round(F.avg(F.col(c).isNotNull().cast("int")), 4).alias(c) for c in feats_inep]))

In [0]:
# Montagem final da wide table + metadados de rastreabilidade
FEATURES_GIRO = [
    "admissoes_lag_1", "admissoes_lag_3", "admissoes_lag_12",
    "admissoes_ma_3", "admissoes_ma_12",
    "desligamentos_lag_1", "desligamentos_ma_3", "saldo_lag_1",
    "salario_mediano_lag_1",
    "admissoes_log_lag_1", "admissoes_log_ma_3", "admissoes_log_ma_12",
    "mes_sin", "mes_cos",
]
FEATURES_FORMACAO = [
    "concluintes_def1", "concluintes_def2", "concluintes_def3",
    "matriculas_def1", "ingressantes_def3", "ingressantes_def5",
    "share_ead_def1", "razao_contratacao_matricula",
]

df_wide = (
    df_formacao
    .select(
        "competencia_mov", "sigla_uf", "codigo_cbo_subgrupo", "admissoes", "desligamentos",
        *(FEATURES_GIRO + FEATURES_FORMACAO),
        F.lit(VERSAO).alias("versao_transformacao"),
        F.lit("0.1.0-provisoria").alias("versao_bridge_origem"),
        F.lit(DT_PROCESSAMENTO).alias("dt_processamento"),
    )
)

# MATERIAL UMA VEZ: o pipeline a montante (silver → janelas → joins INEP) é caro e seria
# recomputado a cada count()/display. Com apenas 47 mil linhas × 32 colunas, o cache é
# barato e evita múltiplos reprocessamentos da silver (~15 min por ação).
df_wide = df_wide.cache()
linhas_total = df_wide.count()  # única ação pesada: materializa o cache

# Validação final — UMA passada sobre o cache (não várias ações de recompute)
# A partir de 2024-01, concluintes_def3 e matriculas_def1 devem estar completas
# (def3 exige t ≥ 2024; ingressantes_def5 exige t ≥ 2026 e pode ter nulos LEGÍTIMOS até lá).
metricas = df_wide.agg(
    F.sum(F.when(F.col("admissoes").isNull(), 1).otherwise(0)).alias("nulos_alvo"),
    F.sum(F.when(
        (F.col("competencia_mov") >= F.lit("2024-01-01")) & (F.col("competencia_mov") < F.lit("2025-01-01"))
        & (F.col("concluintes_def3").isNull() | F.col("matriculas_def1").isNull()),
        1
    ).otherwise(0)).alias("nulos_inep_2024"),
    F.countDistinct("sigla_uf", "codigo_cbo_subgrupo").alias("qtd_series"),
    F.countDistinct("competencia_mov").alias("qtd_meses"),
).collect()[0]

nulos_alvo = metricas["nulos_alvo"]
nulos_inep_2024 = metricas["nulos_inep_2024"]

assert nulos_alvo == 0, "Alvo com nulos — abortando"
assert nulos_inep_2024 == 0, "Features INEP com nulos após o warm-up — abortando"

print(f"Wide table: {linhas_total} linhas, {metricas['qtd_series']} séries, {metricas['qtd_meses']} meses")
print(f"Validações: alvo sem nulos ({nulos_alvo}), features INEP completas a partir de 2024 ({nulos_inep_2024} nulos)")
print("Amostra:")
display(df_wide.where((F.col("competencia_mov") == "2024-06-01") & (F.col("sigla_uf") == "SC")).orderBy("codigo_cbo_subgrupo"))

In [0]:
DDL_FEATURES = f"""
CREATE TABLE IF NOT EXISTS {TABELA_FEATURES} (
    competencia_mov DATE COMMENT 'Competência (mês de referência)',
    sigla_uf STRING COMMENT 'UF (sigla, de-para fixo do código IBGE)',
    codigo_cbo_subgrupo STRING COMMENT 'Subgrupo principal CBO (3 dígitos, cobertura da bridge)',
    admissoes BIGINT COMMENT 'ALVO: admissões no mês (UF × subgrupo)',
    desligamentos BIGINT COMMENT 'Desligamentos no mês (contexto)',
    admissoes_lag_1 BIGINT COMMENT 'Admissões t−1',
    admissoes_lag_3 BIGINT COMMENT 'Admissões t−3',
    admissoes_lag_12 BIGINT COMMENT 'Admissões t−12',
    admissoes_ma_3 DOUBLE COMMENT 'Média móvel de admissões dos 3 meses anteriores (sem o mês corrente)',
    admissoes_ma_12 DOUBLE COMMENT 'Média móvel de admissões dos 12 meses anteriores (sem o mês corrente)',
    desligamentos_lag_1 BIGINT COMMENT 'Desligamentos t−1 (r 0,69 com t+1 na EDA)',
    desligamentos_ma_3 DOUBLE COMMENT 'Média móvel de desligamentos dos 3 meses anteriores',
    saldo_lag_1 BIGINT COMMENT 'Saldo (adm − desl) de t−1',
    salario_mediano_lag_1 DOUBLE COMMENT 'Salário mediano de admissão de t−1 (r 0,68 na EDA)',
    admissoes_log_lag_1 DOUBLE COMMENT 'log1p das admissões t−1',
    admissoes_log_ma_3 DOUBLE COMMENT 'log1p da média móvel 3M',
    admissoes_log_ma_12 DOUBLE COMMENT 'log1p da média móvel 12M',
    mes_sin DOUBLE COMMENT 'Harmônico de mês (sen)',
    mes_cos DOUBLE COMMENT 'Harmônico de mês (cos)',
    concluintes_def1 DOUBLE COMMENT 'Concluintes INEP repartidos (t−1) — bridge uniforme',
    concluintes_def2 DOUBLE COMMENT 'Concluintes INEP repartidos (t−2)',
    concluintes_def3 DOUBLE COMMENT 'Concluintes INEP repartidos (t−3)',
    matriculas_def1 DOUBLE COMMENT 'Matrículas INEP repartidas (t−1)',
    ingressantes_def3 DOUBLE COMMENT 'Ingressantes INEP repartidos (t−3 — pipeline)',
    ingressantes_def5 DOUBLE COMMENT 'Ingressantes INEP repartidos (t−5 — pipeline)',
    share_ead_def1 DOUBLE COMMENT 'Share de ingresso EaD da área (t−1)',
    razao_contratacao_matricula DOUBLE COMMENT 'Admissões UF×subgrupo (t−1) ÷ matrículas repartidas (t−1)',
    versao_transformacao STRING COMMENT 'Versão deste snapshot de features',
    versao_bridge_origem STRING COMMENT 'Versão da bridge CINE×CBO consumida',
    dt_processamento TIMESTAMP COMMENT 'Data/hora da gravação'
)
"""

spark.sql(DDL_FEATURES)

# Gravação idempotente em APPEND — snapshot versionado; consumo filtra pela versão vigente.
# (Sem DROP/overwrite: reexecuções com a mesma versão não duplicam, novas versões anexam histórico.)
versoes_existentes = []
try:
    versoes_existentes = [r["versao_transformacao"] for r in spark.sql(f"SELECT DISTINCT versao_transformacao FROM {TABELA_FEATURES}").collect()]
except Exception:
    pass  # primeira gravação

if VERSAO in versoes_existentes:
    print(f"Versão {VERSAO} já gravada — nada a fazer (idempotente).")
else:
    df_wide.write.mode("append").saveAsTable(TABELA_FEATURES)
    print(f"Features gravadas (append, versão {VERSAO}): {TABELA_FEATURES}")

In [0]:
# Screening OLS — uso conceitual de regressão linear para escolha de features
#
# O QUE É: regressão pooled sobre log1p(admissões), com efeitos fixos de UF e de mês,
# para medir o poder explicativo marginal de cada feature candidata:
#   - coeficiente e p-valor → a feature adiciona informação além dos efeitos fixos?
#   - VIF (variance inflation factor) → multicolinearidade entre candidatas (regra "uma medida por conceito")
# O QUE NÃO É: não é causalidade, não é o modelo final (SARIMA/LightGBM) e não substitui
# a validação temporal em janela expansiva — é uma triagem estatística documentada.

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Painel pequeno (~47 mil linhas): extração controlada para o driver APENAS para o ajuste estatístico.
# A construção distribuída da tabela (células anteriores) continua 100% em Spark.
pdf = (
    spark.table(TABELA_FEATURES)
    .where(F.col("versao_transformacao") == VERSAO)
    .where("competencia_mov >= '2024-01-01'")  # período com TODAS as features INEP preenchidas
    .toPandas()
)
pdf["log_admissoes"] = np.log1p(pdf["admissoes"])
pdf["desl_log_lag_1"] = np.log1p(pdf["desligamentos_lag_1"].clip(lower=0))
pdf["concl_log_def1"] = np.log1p(pdf["concluintes_def1"].fillna(0))
pdf["razao_contratacao_matricula"] = pdf["razao_contratacao_matricula"].fillna(0)

# Linhas com NaN nas features do modelo são excluídas do ajuste (ex.: share_ead nulo quando a
# UF não tem ingressantes na área; salário mediano nulo em mês sem admissões na série)
linhas_antes = len(pdf)
pdf = pdf.dropna(subset=["share_ead_def1", "salario_mediano_lag_1", "desligamentos_lag_1"])

# Mês como categoria (efeito fixo de sazonalidade) — derivado da competência, pois a wide table
# guarda apenas os harmônicos mes_sin/mes_cos
pdf["mes"] = pd.to_datetime(pdf["competencia_mov"]).dt.month
print(f"Painel do screening: {len(pdf)} linhas ({linhas_antes - len(pdf)} excluídas por NaN), período com features completas (2024+)")

FEATURES_SCREENING = [
    "admissoes_log_lag_1", "admissoes_log_ma_3", "admissoes_log_ma_12",
    "desl_log_lag_1", "salario_mediano_lag_1", "mes_sin", "mes_cos",
    "concl_log_def1", "share_ead_def1", "razao_contratacao_matricula",
]
formula = "log_admissoes ~ " + " + ".join(FEATURES_SCREENING) + " + C(sigla_uf) + C(mes)"
modelo = smf.ols(formula, data=pdf).fit()

print(f"R² ajustado do screening (com efeitos fixos de UF e mês): {modelo.rsquared_adj:.4f}")
print(f"Observações: {int(modelo.nobs)}")
print("\nCoeficientes das features candidatas (p-valor < 0,05 passa na triagem):")
display(modelo.params.to_frame("coeficiente").join(modelo.pvalues.to_frame("p_valor")).loc[FEATURES_SCREENING].round(6))

In [0]:
# VIF das features numéricas (multicolinearidade) e gravação idempotente do screening
#
# VIF esperado ALTO entre lags da MESMA medida (admissoes_log_lag_1 × ma_3 × ma_12) — é o que
# motiva a regra "uma medida por conceito": no modelo final escolhe-se UMA representação por família.

# Matriz só com as features numéricas (sem dummies) para o VIF
X_vif = pdf[FEATURES_SCREENING].astype(float).copy()
X_vif.insert(0, "const", 1.0)

df_vif = pd.DataFrame({
    "feature": X_vif.columns[1:],
    "vif": [variance_inflation_factor(X_vif.values, i) for i in range(1, X_vif.shape[1])],
})

df_coefs = pd.DataFrame({
    "coeficiente": modelo.params.loc[FEATURES_SCREENING].values,
    "erro_padrao": modelo.bse.loc[FEATURES_SCREENING].values,
    "estatistica_t": modelo.tvalues.loc[FEATURES_SCREENING].values,
    "p_valor": modelo.pvalues.loc[FEATURES_SCREENING].values,
    "feature": FEATURES_SCREENING,
})

df_screening = (
    df_coefs
    .merge(df_vif, on="feature", how="left")
    .assign(
        decisao_screening=lambda d: np.where(
            (d["p_valor"] < 0.05) & (d["vif"] < 10), "SELECIONADA",
            np.where(d["p_valor"] < 0.05, "SELECIONADA COM COLINEARIDADE", "DESCARTADA NO SCREENING"),
        ),
        criterio="p_valor < 0,05 E VIF < 10 (VIF alto marca colinearidade entre lags da mesma medida)",
        r2_ajustado=modelo.rsquared_adj,
        n_observacoes=int(modelo.nobs),
        versao_transformacao=VERSAO,
        dt_processamento=DT_PROCESSAMENTO,
    )
)

df_screening_spark = spark.createDataFrame(df_screening)

DDL_SCREENING = f"""
CREATE TABLE IF NOT EXISTS {TABELA_SCREENING} (
    feature STRING COMMENT 'Feature candidata avaliada',
    coeficiente DOUBLE COMMENT 'Coeficiente OLS (pooled, efeitos fixos UF e mês)',
    erro_padrao DOUBLE COMMENT 'Erro padrão do coeficiente',
    estatistica_t DOUBLE COMMENT 'Estatística t',
    p_valor DOUBLE COMMENT 'p-valor do teste t',
    vif DOUBLE COMMENT 'Variance Inflation Factor (multicolinearidade)',
    decisao_screening STRING COMMENT 'Veredito da triagem',
    criterio STRING COMMENT 'Critério aplicado',
    r2_ajustado DOUBLE COMMENT 'R² ajustado do modelo de screening',
    n_observacoes BIGINT COMMENT 'Observações do modelo',
    versao_transformacao STRING COMMENT 'Versão do snapshot avaliado',
    dt_processamento TIMESTAMP COMMENT 'Data/hora da gravação'
)
"""
spark.sql(DDL_SCREENING)

versoes_screening = []
try:
    versoes_screening = [r["versao_transformacao"] for r in spark.sql(f"SELECT DISTINCT versao_transformacao FROM {TABELA_SCREENING}").collect()]
except Exception:
    pass

if VERSAO in versoes_screening:
    print(f"Screening da versão {VERSAO} já gravado — nada a fazer (idempotente).")
else:
    df_screening_spark.write.mode("append").saveAsTable(TABELA_SCREENING)
    print(f"Screening gravado (append, versão {VERSAO}): {TABELA_SCREENING}")

display(spark.table(TABELA_SCREENING).where(F.col("versao_transformacao") == VERSAO).orderBy("p_valor"))

## Observações

* **Como consumir** — filtre a wide table por `versao_transformacao` = versão vigente (maior `dt_processamento`). O treino (SARIMA/LightGBM, MLflow, janela expansiva) deve usar o período a partir de 2024-01 para ter todas as features INEP (def3 completa); de 2022 a 2023 as features de formação de defasagem maior ficam nulas por construção (warm-up documentado).
* **OLS e escolha de features** — o screening pooled orienta o corte, com dois cuidados: (1) `admissoes_log_lag_1 × admissoes_log_ma_3 × admissoes_log_ma_12` são a MESMA medida em horizontes diferentes — o VIF alto é esperado e a escolha final fica com UMA representação por conceito; (2) correlação não é causalidade — o screening não valida o modelo, apenas tria candidatos. A decisão final de utilidade continua sendo o erro t+1 na validação em janela expansiva (roteiro B.8/Seção 6).
* **Repartição uniforme** — a formação da área CINE é dividida igualmente entre os subgrupos CBO do par. Evolução natural: ponderar por `confianca` da bridge (ex.: pares ALTA recebem peso maior). A coluna `versao_bridge_origem` garante rastreabilidade dessa decisão.
* **Próximo passo** — notebook de treino: baseline SARIMA (Statsmodels) + challengers LightGBM/Prophet sobre esta tabela, com experimentos no MLflow e validação em janela expansiva.

## Apêndice — Exploração por UF

Células de leitura (nada é gravado) para inspecionar a wide table materializada, sempre filtrando pela versão vigente (`versao_transformacao`). Use os widgets `uf_exemplo` e `subgrupo_exemplo` para navegar.

In [0]:
# Panorama por UF — dimensão do painel, alvo e features-chave
# (leitura da versão vigente; nada é gravado)
df_features = spark.table(TABELA_FEATURES).where(F.col("versao_transformacao") == VERSAO)

print(f"Wide table: {df_features.count()} linhas, {df_features.select('sigla_uf').distinct().count()} UFs")
display(
    df_features
    .groupBy("sigla_uf")
    .agg(
        F.countDistinct("codigo_cbo_subgrupo").alias("subgrupos"),
        F.sum("admissoes").alias("admissoes_total"),
        F.round(F.avg("admissoes"), 1).alias("admissoes_media_serie_mes"),
        F.round(F.avg("salario_mediano_lag_1"), 2).alias("salario_mediano_lag1_medio"),
        F.round(F.avg("share_ead_def1"), 3).alias("share_ead_def1_medio"),
        F.round(F.avg("razao_contratacao_matricula"), 2).alias("razao_contratacao_matricula_media"),
    )
    .orderBy(F.desc("admissoes_total"))
)

In [0]:
# Série mensal completa de uma UF × subgrupo âncora (alvo + lags + formação INEP)
dbutils.widgets.text("uf_exemplo", "SC", "UF da amostra")
dbutils.widgets.text("subgrupo_exemplo", "212", "Subgrupo CBO da amostra")
UF_EXEMPLO = dbutils.widgets.get("uf_exemplo")
SUBGRUPO_EXEMPLO = dbutils.widgets.get("subgrupo_exemplo")

df_amostra = df_features.where(
    (F.col("sigla_uf") == UF_EXEMPLO) & (F.col("codigo_cbo_subgrupo") == SUBGRUPO_EXEMPLO)
)
qtd_meses_amostra = df_amostra.count()
if qtd_meses_amostra == 0:
    print(f"Sem linhas para {UF_EXEMPLO} × subgrupo {SUBGRUPO_EXEMPLO} — ajuste os widgets.")
else:
    print(f"Série {UF_EXEMPLO} × subgrupo {SUBGRUPO_EXEMPLO}: {qtd_meses_amostra} meses")
    display(
        df_amostra.select(
            "competencia_mov", "admissoes", "admissoes_lag_1", "admissoes_ma_12",
            "desligamentos_lag_1", "salario_mediano_lag_1",
            "concluintes_def1", "matriculas_def1", "share_ead_def1", "razao_contratacao_matricula",
        ).orderBy("competencia_mov")
    )

In [0]:
# Comparativo entre UFs para o subgrupo âncora, na última competência disponível
ultima_competencia = df_features.agg(F.max("competencia_mov")).collect()[0][0]
print(f"Última competência da wide table: {ultima_competencia}")
display(
    df_features
    .where((F.col("codigo_cbo_subgrupo") == SUBGRUPO_EXEMPLO) & (F.col("competencia_mov") == ultima_competencia))
    .select(
        "sigla_uf", "admissoes", "admissoes_lag_1", "admissoes_ma_12",
        "salario_mediano_lag_1", "concluintes_def1", "matriculas_def1",
        "razao_contratacao_matricula",
    )
    .orderBy(F.desc("admissoes"))
)